# OfflineMedia — Long-Form Video Generator

Generates videos of **any length** (tested up to 15 min) by chaining Wan2GP clips and stitching them with FFmpeg.

**Run cells top to bottom.** On a free T4 GPU, each 8-second clip takes ~5 min. A 15-min video (~112 clips) needs ~9 hours — use **Colab Pro** for sessions that long.

> **Tip:** Enable Google Drive in Cell 2 to keep models and outputs across restarts.

In [ ]:
# Cell 1 — GPU check
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Runtime → Change runtime type → GPU')
print('GPU:', result.stdout.strip())

In [ ]:
# Cell 2 — Configure storage
# ─────────────────────────────────────────────────
# Set USE_GOOGLE_DRIVE = True to keep models and outputs across Colab restarts.
# With False everything lives in /content and is lost when the session ends.
USE_GOOGLE_DRIVE = False
DRIVE_PATH = 'MyDrive/Wan2GP-data'      # path inside your Google Drive
# ─────────────────────────────────────────────────

In [ ]:
# Cell 3 — Clone / update OfflineMedia repo
import subprocess, sys
from pathlib import Path

REPO_DIR = Path('/content/offlinemedia')
BRANCH = 'claude/scan-repo-chatgpt-review-6nljgh'

if REPO_DIR.exists():
    subprocess.run(['git', 'pull', '--rebase', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    print('Repo updated.')
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH,
         'https://github.com/CAption11/offlinemedia', str(REPO_DIR)],
        check=True,
    )
    print('Repo cloned.')

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Ready.')

In [ ]:
# Cell 4 — Bootstrap Wan2GP (install + start server)
from portable.wan2gp_bootstrap import bootstrap

process = bootstrap(
    use_google_drive=USE_GOOGLE_DRIVE,
    drive_path=DRIVE_PATH,
    share=True,   # prints a public Gradio link — open it in your browser
)
# process is None when an existing healthy server was reused

In [ ]:
# Cell 5 — (Optional) Interactive Gradio UI
# The Gradio link printed above lets you generate videos manually.
# Use it to test prompts before running the automated long-video pipeline.
# When ready for the automated pipeline, run Cell 6 onwards.
print('Open the Gradio link above in your browser to use the interactive UI.')

In [ ]:
# Cell 6 — Define your video
# ─────────────────────────────────────────────────────────────────────────────
# Edit TITLE and PROMPTS.
# Each prompt = one scene clip (~8 seconds by default).
# Add as many prompts as you need — 112 prompts = ~15 minutes of video.
#
# TIP: write prompts as a continuous story; the last frame of each clip
# seeds the next one (continuity mode) so scenes blend smoothly.
# ─────────────────────────────────────────────────────────────────────────────

TITLE = "My Film"

PROMPTS = [
    "A sunrise over misty mountains, golden light piercing through clouds, cinematic wide shot",
    "A hawk soaring through a valley, aerial view, morning light",
    "A river rushing through an ancient forest, slow motion, mist rising",
    # ── add more prompts here ──
]

SCENE_DURATION = 8.0   # seconds per clip
FPS = 8                # frames per second (8 fits T4 VRAM comfortably)
WIDTH = 480
HEIGHT = 272           # 16:9 at 480p — safe on free T4
USE_CONTINUITY = True  # last frame of each clip -> start image of next
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
OUTPUT_DIR = Path('/content/Wan2GP-data/outputs')

n = len(PROMPTS)
est_min = n * SCENE_DURATION / 60
est_hrs = n * 5 / 60
print(f"Plan: {n} scenes, ~{est_min:.1f} min of video, ~{est_hrs:.1f} hrs to generate on T4")

In [ ]:
# Cell 7 — Generate all clips + stitch into final video
from portable.long_video_generator import LongVideoGenerator, plan_from_prompts

plan = plan_from_prompts(
    title=TITLE,
    prompts=PROMPTS,
    scene_duration=SCENE_DURATION,
    fps=FPS,
    width=WIDTH,
    height=HEIGHT,
    use_continuity=USE_CONTINUITY,
    output_dir=OUTPUT_DIR,
)

generator = LongVideoGenerator(plan)
final_video = generator.generate(resume=True)  # resume=True skips already-done clips
print("\nFinal video saved to:", final_video)

In [ ]:
# Cell 8 — Preview in Colab + download link
from IPython.display import Video, display, FileLink
from pathlib import Path

video_path = final_video  # set by Cell 7

if not Path(str(video_path)).exists():
    print("Video not found. Run Cell 7 first.")
else:
    size_mb = Path(str(video_path)).stat().st_size / (1024 * 1024)
    print(f"File: {video_path}  ({size_mb:.1f} MB)")
    display(Video(str(video_path), embed=True, width=720))
    display(FileLink(str(video_path)))

In [ ]:
# Cell 9 — (Optional) Copy to Google Drive
# Only needed when USE_GOOGLE_DRIVE = False in Cell 2.
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
dest = Path('/content/drive/MyDrive/OfflineMedia-Videos') / Path(str(final_video)).name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(str(final_video), str(dest))
print("Saved to Google Drive:", dest)